In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import torch
from pytorch_implementation.prediction.beverse.config import debug_forward_config
from pytorch_implementation.prediction.beverse.metrics import compute_ade_fde, select_best_mode_by_ade
from pytorch_implementation.prediction.beverse.model import BEVerseLite

cfg = debug_forward_config()
model = BEVerseLite(cfg).eval()

batch_size = 2
height, width = 96, 160
img = torch.randn(batch_size, cfg.num_cams, 3, height, width)
img_metas = [{"sample_idx": f"sample_{batch_idx}"} for batch_idx in range(batch_size)]

print(f"Model: {type(model).__name__}")
print(f"Config: {cfg.name}")
print(f"img: {tuple(img.shape)}")

In [ ]:
from typing import Any

def _first_tensor(value: Any):
    """Extract the first tensor from a nested structure."""
    import torch
    if torch.is_tensor(value):
        return value
    if isinstance(value, (tuple, list)):
        for item in value:
            t = _first_tensor(item)
            if t is not None:
                return t
    if isinstance(value, dict):
        for item in value.values():
            t = _first_tensor(item)
            if t is not None:
                return t
    return None

def _iter_tensors(value: Any):
    """Iterate over all tensors in a nested structure."""
    import torch
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _iter_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _iter_tensors(item)

def _register_hook(module, name: str, capture: dict, handles: list) -> None:
    """Register a forward hook that stores output in capture[name]."""
    def _hook(_module, _inputs, output):
        capture[name] = output
    handles.append(module.register_forward_hook(_hook))

def _print_shape(label: str, value) -> None:
    """Print shape of first tensor in value."""
    t = _first_tensor(value)
    if t is not None:
        print(f"  {label}: {tuple(t.shape)}")
    else:
        print(f"  {label}: <no tensor>")

def _check_finite(capture: dict, outputs: dict) -> None:
    """Assert all captured intermediates and outputs are finite."""
    import torch
    for name, value in capture.items():
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in {name}"
    for name, value in outputs.items():
        if value is None:
            continue
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in output {name}"
    print("All intermediate and final tensors are finite.")

# BEVerse Paper-to-Code Study Guide (Prediction)

This note maps BEVerse-style prediction concepts to the pure-PyTorch strict-parity implementation in this repository.

Primary references:
- Paper: [BEVerse: Unified Perception and Prediction in Bird's-Eye View for Vision-Centric Autonomous Driving](https://arxiv.org/abs/2205.09743)
- Reference lineage: [https://github.com/zhangyp15/BEVerse](https://github.com/zhangyp15/BEVerse)
- Pure-PyTorch implementation: `pytorch_implementation/prediction/beverse/`
- Intermediate tensor tests: `tests/prediction/beverse.py`

## 1) Canonical study setup (fixed debug run)

- Config:
  - `debug_forward_config(num_cams=4, embed_dims=96, bev_h=12, bev_w=12, pred_horizon=8, num_modes=3)`
- Input image:
  - `img`: `[B, Ncam, C, H, W] = [2, 4, 3, 96, 160]`
- Metadata:
  - `img_metas`: optional list of sample dictionaries (kept for API parity)

Core dimensions:
- `embed_dims = 96`
- `bev = [Hbev, Wbev] = [12, 12]`
- `pred_horizon = 8`
- `num_modes = 3`
- `future_dt = 0.5 s`

Expected outputs:
- `trajectory_deltas`: `[B, M, T, 2] = [2, 3, 8, 2]`
- `trajectory_preds`: `[B, M, T, 2] = [2, 3, 8, 2]`
- `mode_logits`: `[B, M] = [2, 3]`
- `time_stamps`: `[T] = [8]`

## 2) Symbol dictionary (paper -> code tensors)

- `I_t` (multi-camera image) -> `img` in `BEVerseLite.forward`
- `F_t` (image feature map) -> `camera_feat` from `extract_img_feat`
- `B_t` (BEV tensor) -> `bev_embed`
- `q_t^k` (temporal query token at step `k`) -> `temporal_tokens[:, k]`
- `\Delta p_t^{m,k}` (per-mode displacement) -> `trajectory_deltas[:, m, k]`
- `p_t^{m,k}` (future trajectory point) -> `trajectory_preds[:, m, k]`
- `\pi_t^m` (mode score/probability) -> `mode_logits[:, m]`, `mode_probs[:, m]`
- `\Delta t` (time step) -> `cfg.future_dt`, `time_stamps`

Equation IDs use `E<section>.<index>`.

---

## Chunk 0 - End-to-end forward contract

### Goal
Bind a BEVerse-style prediction path to concrete module calls and tensor outputs.

### Explicit equations
`(E0.1)` Feature and BEV construction:

$$
F_t = \mathrm{CamEncoder}(I_t), \quad
B_t = \mathrm{BEVEncoder}(\mathrm{FuseCam}(F_t))
$$

`(E0.2)` Temporal decoding and multimodal forecasting:

$$
Q_t = \mathrm{TemporalDecoder}(B_t), \quad
\Delta P_t = \mathrm{Head}_{\Delta}(Q_t), \quad
P_t = \mathrm{cumsum}(\Delta P_t)
$$

### Code mapping
- `BEVerseLite.forward` in `pytorch_implementation/prediction/beverse/model.py`
- `TemporalPredictorLite.forward` in `pytorch_implementation/prediction/beverse/temporal.py`
- `TrajectoryHeadLite.forward` in `pytorch_implementation/prediction/beverse/head.py`

### One sanity check
`tests/prediction/beverse.py` verifies full output shapes under debug config.

---

In [ ]:
# Chunk 0: End-to-end forward contract + decode
with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
    decoded_out = model(img, img_metas, decode=True)

print("=== Output shapes ===")
for key, val in outputs.items():
    if torch.is_tensor(val):
        print(f"  {key}: {tuple(val.shape)}")

decoded = decoded_out["decoded"]
print("\n=== Decoded contract ===")
for key, val in decoded.items():
    if torch.is_tensor(val):
        print(f"  {key}: {tuple(val.shape)}")

## Chunk 1 - Multi-camera image encoding and BEV seed

### Goal
Map camera-view CNN features into a compact BEV tensor used by the predictor.

### Explicit equations
`(E1.1)` Camera aggregation:

$$
\bar{F}_t = \frac{1}{N_{cam}}\sum_{c=1}^{N_{cam}} F_t^{(c)}
$$

`(E1.2)` BEV seed resize:

$$
B_t^{seed} = \mathrm{Pool}_{H_{bev},W_{bev}}(\bar{F}_t)
$$

### Code mapping
- `BackboneNeck` in `pytorch_implementation/prediction/beverse/backbone_neck.py`
- camera averaging + adaptive pooling in `BEVerseLite.forward`

### Tensor shape notes
- `camera_feat`: `[B, Ncam, C, Hf, Wf]`
- `bev_seed`: `[B, C, Hbev, Wbev]`
- `bev_embed`: `[B, C, Hbev, Wbev]`

### One sanity check
Tests assert backbone/FPN capture shapes and BEV encoder intermediate shapes.

---

In [ ]:
# Chunk 1: Multi-camera image encoding and BEV seed
capture, handles = {}, []
_register_hook(model.backbone_neck.backbone.stem, "backbone.stem", capture, handles)
for idx, stage in enumerate(model.backbone_neck.backbone.stages):
    _register_hook(stage, f"backbone.stage{idx}", capture, handles)
_register_hook(model.backbone_neck.neck.output_convs[0], "fpn.output0", capture, handles)
_register_hook(model.bev_encoder[0], "bev_encoder.conv0", capture, handles)
_register_hook(model.bev_encoder[3], "bev_encoder.conv1", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Backbone/BEV encoder shapes ===")
for key in sorted(capture.keys()):
    _print_shape(key, capture[key])
_print_shape("camera_feat", outputs["camera_feat"])
_print_shape("bev_embed", outputs["bev_embed"])

## Chunk 2 - Temporal horizon decoding

### Goal
Connect fixed-horizon temporal tokens to the BEV context.

### Explicit equations
`(E2.1)` BEV context pooling:

$$
h_0 = \tanh(W_h \cdot \mathrm{GAP}(B_t))
$$

`(E2.2)` Horizon token decoding:

$$
Q_t = \mathrm{GRU}(E_{time}(1:T), h_0)
$$

### Code mapping
- `TemporalPredictorLite` in `pytorch_implementation/prediction/beverse/temporal.py`

### Tensor shape notes
- `time_embedding`: `[B, T, C]`
- `temporal_tokens`: `[B, T, C]`

### One sanity check
Tests assert `temporal.gru` output shape and finite values.

---

In [ ]:
# Chunk 2: Temporal horizon decoding
capture, handles = {}, []
_register_hook(model.temporal_predictor.time_embedding, "temporal.time_embedding", capture, handles)
_register_hook(model.temporal_predictor.gru, "temporal.gru", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Temporal decoding ===")
_print_shape("temporal.time_embedding", capture["temporal.time_embedding"])
_print_shape("temporal.gru", capture["temporal.gru"])
_print_shape("temporal_tokens", outputs["temporal_tokens"])
print(f"time_stamps: {tuple(outputs['time_stamps'].shape)}")
print(f"time_stamps values: {outputs['time_stamps']}")

## Chunk 3 - Multimodal trajectory and mode heads

### Goal
Map temporal tokens to per-mode XY displacements and mode probabilities.

### Explicit equations
`(E3.1)` Displacement prediction:

$$
\Delta p_t^{m,k} = \tanh\left(W_{\Delta} q_t^k\right)\cdot s_{\max}
$$

`(E3.2)` Trajectory integration:

$$
p_t^{m,k} = \sum_{j=1}^{k}\Delta p_t^{m,j}
$$

`(E3.3)` Mode probabilities:

$$
\pi_t = \mathrm{softmax}(W_{\pi} q_t^T)
$$

### Code mapping
- `TrajectoryHeadLite.forward` in `pytorch_implementation/prediction/beverse/head.py`
- best-mode decode in `BEVerseLite._decode_prediction`

### One sanity check
Tests verify `trajectory_preds == cumsum(trajectory_deltas)` exactly (within tolerance).

---

In [ ]:
# Chunk 3: Multimodal trajectory and mode heads
capture, handles = {}, []
_register_hook(model.trajectory_head.shared[1], "head.shared_fc0", capture, handles)
_register_hook(model.trajectory_head.delta_head, "head.delta", capture, handles)
_register_hook(model.trajectory_head.mode_head, "head.mode", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Trajectory head shapes ===")
_print_shape("head.shared_fc0", capture["head.shared_fc0"])
_print_shape("head.delta", capture["head.delta"])
_print_shape("head.mode", capture["head.mode"])
_print_shape("trajectory_deltas", outputs["trajectory_deltas"])
_print_shape("trajectory_preds", outputs["trajectory_preds"])
_print_shape("mode_logits", outputs["mode_logits"])
_print_shape("mode_probs", outputs["mode_probs"])

reconstructed = outputs["trajectory_deltas"].cumsum(dim=2)
assert torch.allclose(outputs["trajectory_preds"], reconstructed, atol=1e-5, rtol=1e-5)
print("cumsum consistency check passed.")

## Chunk 4 - Prediction metrics and horizon integrity

### Goal
Connect trajectory outputs to ADE/FDE and temporal-axis correctness checks.

### Explicit equations
`(E4.1)` ADE:

$$
\mathrm{ADE} = \frac{1}{T_v}\sum_{k=1}^{T} m_k \| \hat{p}_k - p_k \|_2
$$

`(E4.2)` FDE:

$$
\mathrm{FDE} = \| \hat{p}_{k^*} - p_{k^*} \|_2,\quad k^*=\max\{k \mid m_k=1\}
$$

### Code mapping
- `compute_ade_fde` and `select_best_mode_by_ade` in `pytorch_implementation/prediction/beverse/metrics.py`
- metric smoke test in `tests/prediction/beverse.py`

### One sanity check
Tests assert ADE/FDE are finite/non-negative and `time_stamps` increments by `future_dt`.

---

In [ ]:
# Chunk 4: Prediction metrics and horizon integrity
with torch.no_grad():
    outputs = model(img, img_metas, decode=False)

target = outputs["trajectory_preds"][:, 0] + 0.1
valid_mask = torch.ones((outputs["trajectory_preds"].shape[0], outputs["trajectory_preds"].shape[2]), dtype=target.dtype, device=target.device)
valid_mask[:, -1] = 0.0

best_mode_idx, best_traj, best_ade = select_best_mode_by_ade(
    outputs["trajectory_preds"], target, valid_mask=valid_mask
)
ade, fde = compute_ade_fde(best_traj, target, valid_mask=valid_mask)

print("=== Metric smoke ===")
print(f"best_mode_idx shape: {tuple(best_mode_idx.shape)}")
print(f"best_traj shape: {tuple(best_traj.shape)}")
print(f"best_ade: {best_ade}")
print(f"ADE: {ade}")
print(f"FDE: {fde}")

## 3) Dataflow diagram

```mermaid
flowchart LR
    img["img BxNcamx3xHxW"] --> enc[BackboneNeck]
    enc --> cam["camera_feat BxNcamxCxHfxWf"]
    cam --> avg["mean over cameras"]
    avg --> pool["adaptive pool to BEV"]
    pool --> bev[BEV encoder convs]
    bev --> temp[TemporalPredictorLite GRU]
    temp --> head[TrajectoryHeadLite]
    head --> delta["trajectory_deltas BxMxTx2"]
    delta --> csum["cumsum over T"]
    csum --> traj["trajectory_preds BxMxTx2"]
    head --> mode["mode_probs BxM"]
```

## 4) One end-to-end tensor trace

1. Input `img [2, 4, 3, 96, 160]`.
2. Backbone+FPN returns one level, reshaped to `camera_feat [2, 4, 96, 6, 10]`.
3. Camera-average + pooling produce `bev_seed [2, 96, 12, 12]`.
4. BEV encoder outputs `bev_embed [2, 96, 12, 12]`.
5. Temporal GRU decodes `temporal_tokens [2, 8, 96]`.
6. Head predicts:
   - `trajectory_deltas [2, 3, 8, 2]`
   - `trajectory_preds [2, 3, 8, 2]`
   - `mode_probs [2, 3]`.
7. Decode picks argmax mode:
   - `best_trajectory [2, 8, 2]`.

## 5) Study drills (self-check questions)

1. Why does this implementation predict displacements (`\Delta p`) before cumulative positions (`p`)?
2. Which tensor corresponds to paper symbol `q_t^k`?
3. Which axis in `trajectory_preds` is time, and how is it validated?
4. Why use a learned time embedding before GRU decoding?
5. How do ADE/FDE differ when masks hide the final step?

## 6) Practical reading order

1. Read Chunk 0 for the overall contract.
2. Read Chunk 1 to map image features into BEV.
3. Read Chunk 2 and Chunk 3 for temporal decoding and multimodal outputs.
4. Read Chunk 4 and then inspect `tests/prediction/beverse.py`.

## 7) Strict parity notes and pure-PyTorch replacements

- Behavioral parity is pinned to frozen BEVerse anchors in `study/markdown/strict_parity_anchor_manifest.md`.
- Multi-task contract (`map`, `3dod`, `motion`) is preserved with strict task-output ordering and motion decode semantics.
- Temporal neck behavior keeps sequence validity and time-index contracts, including monotonic horizon checks.
- Framework/runtime dependencies are replaced with pure PyTorch modules while preserving trajectory mode semantics.

In [ ]:
# Final finite check with major hooks
capture, handles = {}, []
_register_hook(model.backbone_neck.backbone.stem, "backbone.stem", capture, handles)
for idx, stage in enumerate(model.backbone_neck.backbone.stages):
    _register_hook(stage, f"backbone.stage{idx}", capture, handles)
_register_hook(model.backbone_neck.neck.output_convs[0], "fpn.output0", capture, handles)
_register_hook(model.bev_encoder[0], "bev_encoder.conv0", capture, handles)
_register_hook(model.bev_encoder[3], "bev_encoder.conv1", capture, handles)
_register_hook(model.temporal_predictor.time_embedding, "temporal.time_embedding", capture, handles)
_register_hook(model.temporal_predictor.gru, "temporal.gru", capture, handles)
_register_hook(model.trajectory_head.shared[1], "head.shared_fc0", capture, handles)
_register_hook(model.trajectory_head.delta_head, "head.delta", capture, handles)
_register_hook(model.trajectory_head.mode_head, "head.mode", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

_check_finite(capture, outputs)